Purpose: Look at time-structured gene results from maSigPro, comparing input data with 4 reps and that with 3 reps.<br>
Author: Anna Pardo<br>
Date initiated: Apr. 13, 2026

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

In [12]:
# load CAM gene annotations
cam = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_insilico_genome/degs_downstream/camgenes_Ya_Yf_orthology_synteny.txt",
                  sep="\t",header="infer")

In [42]:
# define a function to load & summarize the data for a given filepath
def load_sum_file(filepath,phys):
    gtc = pd.read_csv(filepath,sep="\t",header="infer")
    if "Yg" in filepath:
        sg = []
        for i in gtc["GeneID"]:
            if i.startswith("Yucal"):
                sg.append("Ya")
            else:
                sg.append("Yf")
        gtc["subgenome"] = sg
        sum1 = gtc.groupby(["cluster","subgenome"]).count().reset_index().rename(columns={"GeneID":"n_genes_"+phys})
        gtcam = cam.merge(gtc)
        camsum = gtcam.groupby(["cluster","subgenome"]).count().reset_index()[["cluster","subgenome","GeneID"]].rename(columns={"GeneID":"n_genes_"+phys})
    else:
        sum1 = gtc.groupby("cluster").count().reset_index().rename(columns={"GeneID":"n_genes_"+phys})
        gtcam = cam.merge(gtc)
        camsum = gtcam.groupby("cluster").count().reset_index()[["cluster","GeneID"]].rename(columns={"GeneID":"n_genes_"+phys})
    
    d = {"clusters":gtc,"CAMgenes":gtcam,"all_summary":sum1,"CAM_summary":camsum}
    return d

In [33]:
# load 4-reps results
sumdict_4reps = {}
for f in os.listdir("./"):
    if ("4reps" in f) and (f.endswith(".txt")):
        spphys = f.strip().split("_4")[0]
        phys = spphys.split("g_")[1]
        sumdict_4reps[phys] = load_sum_file(os.path.join("./",f),phys)

In [64]:
# load 3-reps results
sumdict_3reps = {}
for f in os.listdir("./"):
    if ("3reps" in f) and (f.endswith(".txt")) and (f.startswith("Yg")):
        spphys = f.strip().split("_3")[0]
        phys = spphys.split("g_")[1]
        sumdict_3reps[phys] = load_sum_file(os.path.join("./",f),phys)

In [65]:
sumdict_3reps.keys()

dict_keys(['possible_fac_CAM', 'facultative_CAM', 'C3'])

In [66]:
sumdict_3reps["facultative_CAM"]["all_summary"].groupby("subgenome").sum()

,cluster,n_genes_facultative_CAM
subgenome,,
Ya,21,7982
Yf,21,6952


In [67]:
7982+6952

14934

In [44]:
# load Yf results
yfsum = load_sum_file("./Yf_3reps_masigpro_clusters.txt","Yf")

In [47]:
# load Ya results (from a previous run)
yasum = load_sum_file("/home/leviathan22/Yucca_genomics/rna_insilico_genome/masigpro_results/Ya_hclust_k6_clusters.txt","Ya")

In [49]:
yasum["all_summary"]["n_genes_Ya"].sum()

4915

In [50]:
# load Yf results (from previous run, mostly 4 reps)
yfold = load_sum_file("/home/leviathan22/Yucca_genomics/rna_insilico_genome/masigpro_results/Yf_hclust_k6_clusters.txt","Yf_old")

In [51]:
yfold["all_summary"]["n_genes_Yf_old"].sum()

6021

In [54]:
yfsum["CAMgenes"]

,GeneID,Orthogroup,Pathway,gene_name,gene_abbr,gene_abbr_unique,subgenome,cluster
0,YufilH1032169m.g,OG0001083,CAM-dark,phosphoenolpyruvate carboxylase kinase,PPCK,Yf_PPCK_2,Yf,3
1,YufilH1062820m.g,OG0002410,CAM-light,"pyruvate, phosphate dikinase",PPDK,Yf_PPDK_1,Yf,2


In [55]:
yfold["CAMgenes"]

,GeneID,Orthogroup,Pathway,gene_name,gene_abbr,gene_abbr_unique,subgenome,cluster
0,YufilH1042055m.g,OG0004406,CAM-dark,beta-carbonic anhydrase,bCA5,Yf_bCA5_4,Yf,3
1,YufilH1044442m.g,OG0008119,CAM-dark,phosphoenolpyruvate carboxylase (bacterial-type),BPPC,Yf_BPPC,Yf,4
2,YufilH1062820m.g,OG0002410,CAM-light,"pyruvate, phosphate dikinase",PPDK,Yf_PPDK_1,Yf,3


In [56]:
sumdict_3reps["C3"]["CAMgenes"]

,GeneID,Orthogroup,Pathway,gene_name,gene_abbr,gene_abbr_unique,subgenome,cluster
0,Yucal.04G064300.v2.1,OG0002899,CAM-dark,NAD-dependent malate dehydrogenase (chloroplas...,NAD-MDH-cp,Ya_NAD-MDH-cp_2,Ya,6
1,Yucal.07G108400.v2.1,OG0008119,CAM-dark,phosphoenolpyruvate carboxylase (bacterial-type),BPPC,Ya_BPPC_2,Ya,3
2,Yucal.07G108500.v2.1,OG0008119,CAM-dark,phosphoenolpyruvate carboxylase (bacterial-type),BPPC,Ya_BPPC_3,Ya,3
3,Yucal.06G042800.v2.1,OG0000834,CAM-dark,phosphoenolpyruvate carboxylase (plant-type),PPPC,Ya_PPPC_2,Ya,1
4,Yucal.07G056400.v2.1,OG0000834,CAM-dark,phosphoenolpyruvate carboxylase (plant-type),PPPC,Ya_PPPC_3,Ya,5
5,Yucal.21G047000.v2.1,OG0000834,CAM-dark,phosphoenolpyruvate carboxylase (plant-type),PPPC,Ya_PPPC_4,Ya,5
6,Yucal.02G133400.v2.1,OG0000793,CAM-light,NADP-dependent malic enzyme,NADP-ME,Ya_NADP-ME_2,Ya,1
7,Yucal.12G110300.v2.1,OG0002410,CAM-light,"pyruvate, phosphate dikinase",PPDK,Ya_PPDK_1,Ya,3
8,Yucal.12G110500.v2.1,OG0002410,CAM-light,"pyruvate, phosphate dikinase",PPDK,Ya_PPDK_2,Ya,3
9,Yucal.12G110700.v2.1,OG0002410,CAM-light,"pyruvate, phosphate dikinase",PPDK,Ya_PPDK_3,Ya,3


In [57]:
sumdict_4reps["C3"]["CAMgenes"]

,GeneID,Orthogroup,Pathway,gene_name,gene_abbr,gene_abbr_unique,subgenome,cluster
0,Yucal.01G165600.v2.1,OG0001578,CAM-dark,beta-carbonic anhydrase,bCA1234,Ya_bCA1234_1,Ya,4
1,Yucal.04G064300.v2.1,OG0002899,CAM-dark,NAD-dependent malate dehydrogenase (chloroplas...,NAD-MDH-cp,Ya_NAD-MDH-cp_2,Ya,5
2,Yucal.07G108300.v2.1,OG0008119,CAM-dark,phosphoenolpyruvate carboxylase (bacterial-type),BPPC,Ya_BPPC_1,Ya,2
3,Yucal.07G108400.v2.1,OG0008119,CAM-dark,phosphoenolpyruvate carboxylase (bacterial-type),BPPC,Ya_BPPC_2,Ya,2
4,Yucal.07G108500.v2.1,OG0008119,CAM-dark,phosphoenolpyruvate carboxylase (bacterial-type),BPPC,Ya_BPPC_3,Ya,2
5,Yucal.06G042800.v2.1,OG0000834,CAM-dark,phosphoenolpyruvate carboxylase (plant-type),PPPC,Ya_PPPC_2,Ya,4
6,Yucal.01G137500.v2.1,OG0000793,CAM-light,NADP-dependent malic enzyme,NADP-ME,Ya_NADP-ME_1,Ya,4
7,Yucal.02G133400.v2.1,OG0000793,CAM-light,NADP-dependent malic enzyme,NADP-ME,Ya_NADP-ME_2,Ya,4
8,Yucal.06G106600.v2.1,OG0000793,CAM-light,NADP-dependent malic enzyme,NADP-ME,Ya_NADP-ME_3,Ya,2
9,Yucal.21G055300.v2.1,OG0000793,CAM-light,NADP-dependent malic enzyme,NADP-ME,Ya_NADP-ME_5,Ya,3


In [58]:
set(list(sumdict_3reps["C3"]["CAMgenes"]["gene_abbr_unique"])).intersection(set(list(sumdict_4reps["C3"]["CAMgenes"]["gene_abbr_unique"])))

{'Ya_BPPC_2',
 'Ya_BPPC_3',
 'Ya_NAD-MDH-cp_2',
 'Ya_NADP-ME_2',
 'Ya_PPDK_1',
 'Ya_PPDK_2',
 'Ya_PPDK_3',
 'Ya_PPPC_2',
 'Yf_BPPC',
 'Yf_NADP-ME_6',
 'Yf_PPDK_1'}

## Yf CAM genes: plot TPM patterns for only the samples that went into the 3-rep run (vs. the total run) for all CAM genes identified as TS in either or both runs

In [63]:
# list of relevant genes
yfcg = ["YufilH1032169m.g","YufilH1062820m.g","YufilH1042055m.g","YufilH1044442m.g"]

In [70]:
# load sample list from 3-rep run of Yf (from counts matrix used as input)
yf3rsamp = list(pd.read_csv("./inputdata_3reps/YfH1_counts_withmd_3reps_random.txt",sep="\t",header="infer",
                      usecols=["sample_name"])["sample_name"])

In [71]:
# load Yf TPM
yftpm = pd.read_csv("/home/leviathan22/Yucca_genomics/rna_reanalysis_30-Jun-2025/TPM/YfilH1_TPM_withmd_July2025.txt",
                   sep="\t",header="infer")

In [72]:
# subset to just the 3-reps sample set
tpm3r = yftpm[yftpm["sample_name"].isin(yf3rsamp)]

In [77]:
yftpm.head()

,sample_name,genotype,time,treat,ZT,species,YufilH1000001m.g,YufilH1000003m.g,YufilH1000007m.g,YufilH1000011m.g,...,YufilH1095122m.g,YufilH1095123m.g,YufilH1095125m.g,YufilH1095126m.g,YufilH1095128m.g,YufilH1095131m.g,YufilH1095132m.g,YufilH1095134m.g,YufilH1095146m.g,YufilH1095147m.g
0,X73,2.0,4.0,D,13.0,filamentosa,0.0,69.209532,18.118388,14.647685,...,0.000000,8.398514,73.738130,2.102850,19.402193,20.453314,8.251751,16.330160,0.0,0.000000
1,X74,2.0,5.0,D,17.0,filamentosa,0.0,89.139056,20.331585,15.566820,...,0.000000,6.640828,72.812930,2.554231,22.485396,15.573745,7.444941,19.426909,0.0,1.679625
2,X64,2.0,2.0,W,5.0,filamentosa,0.0,40.700085,8.461118,16.711576,...,0.000000,11.801415,102.629400,1.869051,29.032564,16.676626,5.797593,8.062388,0.0,0.000000
3,X101,27.0,3.0,D,9.0,filamentosa,0.0,48.314400,23.459476,18.753399,...,0.000000,12.020853,74.562204,2.408896,19.987492,27.106460,5.948658,7.219606,0.0,1.085844
4,X102,27.0,1.0,D,1.0,filamentosa,0.0,58.172755,23.191529,14.185471,...,0.575726,6.796891,51.324403,1.323844,22.161740,4.797914,2.791854,27.798525,0.0,0.000000


In [103]:
# make a function to plot a given gene comparing datasets
def plot_gene(gid,outdir):
    # get gene name
    gname = cam.loc[cam["GeneID"]==gid,"gene_abbr_unique"].iloc[0]
    
    alltpm = yftpm[["sample_name","treat","ZT",gid]]
    r3tpm = tpm3r[["sample_name","treat","ZT",gid]]
    
    fig,ax = plt.subplots(nrows=1,ncols=2,figsize=(20,10),sharey=True)
    t = np.arange(0,25)
    xthresh = 12
    sns.lineplot(ax=ax[0],data=alltpm,x="ZT",y=gid,hue="treat")
    sns.lineplot(ax=ax[1],data=r3tpm,x="ZT",y=gid,hue="treat")
    
    ax[0].fill_between(t,0,1,where=t>=xthresh,color="gray",alpha=0.3,transform=ax[0].get_xaxis_transform())
    ax[1].fill_between(t,0,1,where=t>=xthresh,color="gray",alpha=0.3,transform=ax[1].get_xaxis_transform())
    ax[0].set_xlim(0,24)
    ax[1].set_xlim(0,24)
    ax[0].tick_params(axis="both",which="major",labelsize=16)
    ax[1].tick_params(axis="both",which="major",labelsize=16)
    ax[0].set_xticks(ticks=list(alltpm["ZT"].unique()))
    ax[1].set_xticks(ticks=list(r3tpm["ZT"].unique()))
    ax[0].set_xlabel("")
    ax[1].set_xlabel("")
    ax[0].set_ylabel("TPM",fontsize=16)
    ax[0].set_title(gname+" Expression with 4 reps",fontsize=22)
    ax[1].set_title(gname+" Expression with 3 reps",fontsize=22)
    
    fig.tight_layout()
    
    plt.savefig(os.path.join(outdir,"Yf_repscompare_plot_"+gname+".pdf"),dpi=300,bbox_inches="tight")

In [106]:
for i in yfcg:
    plot_gene(i,"./troubleshooting_plots/")
    plt.cla()
    plt.clf()

<Figure size 1440x720 with 0 Axes>

<Figure size 1440x720 with 0 Axes>

<Figure size 1440x720 with 0 Axes>

<Figure size 1440x720 with 0 Axes>